In [5]:
import numpy as np
import pandas as pd

In [6]:
import torch


In [7]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split

In [8]:
df=pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [ ]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [9]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [10]:
x_train,x_test,y_train,y_test=train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [11]:
from sklearn.preprocessing import StandardScaler
x_train_scaled=StandardScaler().fit_transform(x_train)
x_test_scaled=StandardScaler().fit_transform(x_test)

In [12]:
encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

In [13]:
x_train_tensor=torch.from_numpy(x_train_scaled).float()
x_test_tensor=torch.from_numpy(x_test_scaled).float()
y_train_tensor=torch.from_numpy(y_train).float()
y_test_tensor=torch.from_numpy(y_test).float()

In [14]:
class NeuralNetwork():
    def __init__(self,X):
      self.weights=torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
      self.bias=torch.zeros(1,dtype=torch.float64,requires_grad=True)
    def forward(self,X):
      return torch.sigmoid(torch.matmul(X,self.weights)+self.bias)
    def evaluvate_loss(self,y_pred,y_true):
      y_pred=torch.clamp(y_pred,min=1e-7,max=1-1e-7)
      return -(y_true*torch.log(y_pred)+(1-y_true)*torch.log(1-y_pred))
    def loss_function(self,y_pred,y_true):
      return torch.mean(self.evaluvate_loss(y_pred,y_true))




In [15]:
learning_rate=0.1
epochs=100

In [ ]:
model=NeuralNetwork(x_train_tensor)

In [ ]:
model.weights.shape

torch.Size([30, 1])

In [ ]:
for i in range(epochs):
  y_pred=model.forward(x_train_tensor)
  loss= model.loss_function(y_pred,y_train_tensor)
  loss.backward()
  with torch.no_grad():
    model.weights-=learning_rate*model.weights.grad
    model.bias-=learning_rate*model.bias.grad
  model.weights.grad.zero_()
  model.bias.grad.zero_()
  print(f'Epoch:{i+1}, Loss:{loss.item()}')


Epoch:1, Loss:3.638308964109722
Epoch:2, Loss:3.246955052531742
Epoch:3, Loss:3.1026504622506166
Epoch:4, Loss:2.954985295571865
Epoch:5, Loss:2.8016873385629877
Epoch:6, Loss:2.6435640136819005
Epoch:7, Loss:2.4780753447763764
Epoch:8, Loss:2.3035413792642965
Epoch:9, Loss:2.12153610561008
Epoch:10, Loss:1.9374479797956496
Epoch:11, Loss:1.7603602491912087
Epoch:12, Loss:1.587876858774032
Epoch:13, Loss:1.4235075280327096
Epoch:14, Loss:1.274027867453158
Epoch:15, Loss:1.1421313420819215
Epoch:16, Loss:1.0301965998538412
Epoch:17, Loss:0.9397951209156465
Epoch:18, Loss:0.8709179716696485
Epoch:19, Loss:0.8211373064903171
Epoch:20, Loss:0.7863337321895484
Epoch:21, Loss:0.7623889271487845
Epoch:22, Loss:0.7459256488490249
Epoch:23, Loss:0.7344230053267699
Epoch:24, Loss:0.7261368991896596
Epoch:25, Loss:0.7199306940223567


In [24]:
from sklearn.metrics import accuracy_score

In [ ]:
with torch.no_grad():
  y_pred=model.forward(x_test_tensor)
  y_pred=(y_pred>0.9).float()
  print(accuracy_score(y_pred,y_test_tensor))


0.9035087719298246


In [16]:
import torch.nn as nn

In [17]:
class NewNeuralNetwork(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network=nn.Sequential(nn.Linear(num_features,128),
    nn.ReLU()
    ,nn.Linear(128,64),
    nn.ReLU()
    ,nn.Linear(64,1),
    nn.Sigmoid()
    )
  def forward(self,X):
    return self.network(X)


In [18]:
loss_function=nn.BCELoss()

In [2]:
from torch.utils.data import Dataset,DataLoader

In [3]:
class Customdataset(Dataset):
  def __init__(self,x,y):
    self.x=x
    self.y=y
  def __len__(self):
    return len(self.x)
  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]

In [19]:
x_train_dataset=Customdataset(x_train_tensor,y_train_tensor)
x_test_dataset=Customdataset(x_test_tensor,y_test_tensor)

In [20]:
x_train_loader=DataLoader(x_train_dataset,batch_size=32,shuffle=True)
x_test_loader=DataLoader(x_test_dataset,batch_size=32,shuffle=True)

In [21]:
model=NewNeuralNetwork(x_train_tensor.shape[1])
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
for i in range(epochs):
  for x_batch,y_batch in x_train_loader:
    y_pred=model(x_batch)
    loss=loss_function(y_pred,y_batch.reshape(-1,1))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
  print(f'Epoch:{i+1}, Loss:{loss.item()}')

Epoch:1, Loss:0.5209528803825378
Epoch:2, Loss:0.2833264470100403
Epoch:3, Loss:0.11979774385690689
Epoch:4, Loss:0.13035903871059418
Epoch:5, Loss:0.12987352907657623
Epoch:6, Loss:0.4250948131084442
Epoch:7, Loss:0.27699029445648193
Epoch:8, Loss:0.04997577518224716
Epoch:9, Loss:0.0031666974537074566
Epoch:10, Loss:0.03323253616690636
Epoch:11, Loss:0.043769411742687225
Epoch:12, Loss:0.007908233441412449
Epoch:13, Loss:0.00421613734215498
Epoch:14, Loss:0.011507212184369564
Epoch:15, Loss:0.0023985723964869976
Epoch:16, Loss:0.011214518919587135
Epoch:17, Loss:0.02050955407321453
Epoch:18, Loss:0.001116994651965797
Epoch:19, Loss:0.0036495733074843884
Epoch:20, Loss:0.003578619798645377
Epoch:21, Loss:0.0030846851877868176
Epoch:22, Loss:0.018468761816620827
Epoch:23, Loss:0.03133818879723549
Epoch:24, Loss:0.0400649830698967
Epoch:25, Loss:0.3277653753757477
Epoch:26, Loss:0.4288092255592346
Epoch:27, Loss:0.001087008509784937
Epoch:28, Loss:0.14151827991008759
Epoch:29, Loss:0.13

In [26]:
model.eval()
accuracy_list=[]
with torch.no_grad():
  for x_batch,y_batch in x_test_loader:
    y_pred=model(x_batch)
    y_pred=(y_pred>0.5).float()
    accuracy=accuracy_score(y_pred,y_batch.reshape(-1,1))
    accuracy_list.append(accuracy)
print(sum(accuracy_list)/len(accuracy_list))

0.96875
